In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, GRU
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
import warnings
import ipywidgets as widgets
from IPython.display import display, clear_output

2025-04-25 17:06:58.865370: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/islem/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pd.read_csv("/Users/islem/Desktop/covid 19/covid_19_clean_complete.csv")

In [3]:
df['Date'] = pd.to_datetime(df['Date'])

In [4]:
# Fill NA values
df['Province/State'].fillna('', inplace=True)
df['Confirmed'].fillna(0, inplace=True)
df['Deaths'].fillna(0, inplace=True)
df['Recovered'].fillna(0, inplace=True)
df['Active'].fillna(0, inplace=True)

# Display the first few rows
print("Dataset overview:")
display(df.head())

Dataset overview:


/var/folders/6r/l1bqm8tx1rd1qtbqt5mtd1c00000gn/T/ipykernel_20488/2130088006.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Province/State'].fillna('', inplace=True)
/var/folders/6r/l1bqm8tx1rd1qtbqt5mtd1c00000gn/T/ipykernel_20488/2130088006.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always 

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,,Afghanistan,33.0000,65.0000,2020-01-22,0,0,0,0,Eastern Mediterranean
1,,Albania,41.1533,20.1683,2020-01-22,0,0,0,0,Europe
2,,Algeria,28.0339,1.6596,2020-01-22,0,0,0,0,Africa
3,,Andorra,42.5063,1.5218,2020-01-22,0,0,0,0,Europe
4,,Angola,-11.2027,17.8739,2020-01-22,0,0,0,0,Africa


In [5]:
# Get unique countries for dropdown
unique_countries = sorted(df['Country/Region'].unique())

# CELL 2: User Input Interface
# Create widgets for user input
country_dropdown = widgets.Dropdown(
    options=unique_countries,
    description='Country:',
    style={'description_width': 'initial'}
)

population_input = widgets.IntText(
    value=1000000,
    description='Population:',
    style={'description_width': 'initial'}
)

# Create date picker for prediction target
latest_date = df['Date'].max()
prediction_date = widgets.DatePicker(
    description='Predict for date:',
    value=latest_date + timedelta(days=7),
    style={'description_width': 'initial'}
)

# Add option to use custom SIR parameters
use_custom_sir = widgets.Checkbox(
    value=False,
    description='Use custom SIR values?',
    style={'description_width': 'initial'}
)

# SIR input fields (initially hidden)
susceptible_input = widgets.IntText(
    value=900000,
    description='Susceptible:',
    disabled=True,
    style={'description_width': 'initial'}
)

infected_input = widgets.IntText(
    value=5000,
    description='Infected:',
    disabled=True,
    style={'description_width': 'initial'}
)

recovered_input = widgets.IntText(
    value=95000,
    description='Recovered:',
    disabled=True,
    style={'description_width': 'initial'}
)

In [6]:
# Function to toggle SIR input fields
def toggle_sir_inputs(change):
    susceptible_input.disabled = not change['new']
    infected_input.disabled = not change['new']
    recovered_input.disabled = not change['new']

use_custom_sir.observe(toggle_sir_inputs, names='value')

# Run button
run_button = widgets.Button(
    description='Run Analysis',
    button_style='success',
    tooltip='Click to run the analysis'
)

# Layout all widgets
input_widgets = widgets.VBox([
    country_dropdown,
    population_input,
    prediction_date,
    use_custom_sir,
    susceptible_input,
    infected_input,
    recovered_input,
    run_button
])

# Display the input interface
display(input_widgets)

# CELL 3: Main Analysis Function
# Function to run the analysis when the button is clicked

In [7]:
def run_analysis(b):
    clear_output(wait=True)
    display(input_widgets)
    
    # Get selected values
    country = country_dropdown.value
    population = population_input.value
    target_date = prediction_date.value
    
    # Prepare country-specific data
    print(f"\nAnalyzing COVID-19 data for {country}...")
    country_df = df[df['Country/Region'] == country].copy()
    
    # Aggregate by date if multiple provinces
    daily_data = country_df.groupby('Date').agg({
        'Confirmed': 'sum',
        'Deaths': 'sum',
        'Recovered': 'sum',
        'Active': 'sum'
    }).reset_index()
    
    # Sort by date
    daily_data = daily_data.sort_values('Date')
    
    # Calculate daily new cases (diff)
    daily_data['New_Confirmed'] = daily_data['Confirmed'].diff().fillna(0)
    daily_data['New_Deaths'] = daily_data['Deaths'].diff().fillna(0)
    daily_data['New_Recovered'] = daily_data['Recovered'].diff().fillna(0)
    
    # Ensure no negative values in new cases (data errors)
    daily_data['New_Confirmed'] = daily_data['New_Confirmed'].clip(lower=0)
    daily_data['New_Deaths'] = daily_data['New_Deaths'].clip(lower=0)
    daily_data['New_Recovered'] = daily_data['New_Recovered'].clip(lower=0)
    
    # Display the processed data
    print(f"Data prepared for {country}. Latest available date: {daily_data['Date'].max().strftime('%Y-%m-%d')}")
    
    # Set up SIR model initial conditions
    if use_custom_sir.value:
        S0 = susceptible_input.value
        I0 = infected_input.value
        R0 = recovered_input.value
        print(f"Using custom SIR values - S: {S0}, I: {I0}, R: {R0}")
    else:
        I0 = daily_data['Active'].iloc[-1]
        R0 = daily_data['Recovered'].iloc[-1] + daily_data['Deaths'].iloc[-1]
        S0 = population - I0 - R0
        print(f"Using latest data SIR values - S: {S0}, I: {I0}, R: {R0}")
    
    initial_conditions = [S0, I0, R0]
    
    # Visualize the historical data
    plt.figure(figsize=(14, 8))
    plt.plot(daily_data['Date'], daily_data['Confirmed'], 'b-', label='Confirmed')
    plt.plot(daily_data['Date'], daily_data['Deaths'], 'r-', label='Deaths')
    plt.plot(daily_data['Date'], daily_data['Recovered'], 'g-', label='Recovered')
    plt.plot(daily_data['Date'], daily_data['Active'], 'y-', label='Active')
    plt.title(f'COVID-19 Statistics for {country}')
    plt.xlabel('Date')
    plt.ylabel('Cases')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # SIR Model Implementation
    print("\nFitting SIR model to historical data...")
    
    # Define SIR model differential equations
    def sir_model(t, y, beta, gamma, N):
        """SIR model differential equations."""
        S, I, R = y
        dSdt = -beta * S * I / N
        dIdt = beta * S * I / N - gamma * I
        dRdt = gamma * I
        return [dSdt, dIdt, dRdt]
    
    # Time points
    t = np.arange(len(daily_data))
    
    # Actual data for fitting
    actual_infected = daily_data['Active'].values
    actual_recovered = daily_data['Recovered'].values + daily_data['Deaths'].values
    
    # Function to minimize
    def loss_function(params):
        beta, gamma = params
        solution = solve_ivp(
            fun=lambda t, y: sir_model(t, y, beta, gamma, population),
            t_span=[0, max(t)],
            y0=initial_conditions,
            t_eval=t,
            method='RK45'
        )
        
        S, I, R = solution.y
        loss = np.mean((I - actual_infected)**2) + np.mean((R - actual_recovered)**2)
        return loss
    
    # Initial guess for parameters
    initial_params = [0.5, 0.1]  # beta, gamma
    
    # Optimize to find the best parameters
    bounds = [(0.01, 2), (0.01, 1)]  # bounds for beta and gamma
    result = minimize(loss_function, initial_params, bounds=bounds, method='L-BFGS-B')
    beta_opt, gamma_opt = result.x
    
    print(f"SIR Model Parameters:")
    print(f"Beta (infection rate): {beta_opt:.4f}")
    print(f"Gamma (recovery rate): {gamma_opt:.4f}")
    print(f"R0 (basic reproduction number): {beta_opt/gamma_opt:.4f}")
    
    # Simulate with optimized parameters
    solution_opt = solve_ivp(
        fun=lambda t, y: sir_model(t, y, beta_opt, gamma_opt, population),
        t_span=[0, max(t)],
        y0=initial_conditions,
        t_eval=t,
        method='RK45'
    )
    
    S_opt, I_opt, R_opt = solution_opt.y
    
    # Plot SIR model fit
    plt.figure(figsize=(14, 8))
    plt.plot(daily_data['Date'], actual_infected, 'b.', alpha=0.6, label='Actual Infected')
    plt.plot(daily_data['Date'], I_opt, 'b-', label='SIR Model Infected')
    plt.plot(daily_data['Date'], actual_recovered, 'r.', alpha=0.6, label='Actual Recovered+Deaths')
    plt.plot(daily_data['Date'], R_opt, 'r-', label='SIR Model Recovered')
    plt.title(f'SIR Model Fit for {country}')
    plt.xlabel('Date')
    plt.ylabel('Cases')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Deep Learning Model Preparation
    print("\nPreparing deep learning model...")
    
    # Prepare features for deep learning
    features = ['Confirmed', 'Deaths', 'Recovered', 'Active', 
                'New_Confirmed', 'New_Deaths', 'New_Recovered']
    
    data = daily_data[features].values
    
    # Scale the data
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data)
    
    # Define sequence length
    seq_length = 14  # 14 days for prediction
    
    # Create sequences for LSTM training
    X, y = [], []
    for i in range(len(scaled_data) - seq_length):
        X.append(scaled_data[i:i+seq_length])
        y.append(scaled_data[i+seq_length, 3])  # Target is Active cases (index 3)
    
    X, y = np.array(X), np.array(y)
    
    # Train-test split
    train_size = int(len(X) * 0.8)
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    
    print(f"Training deep learning model with {len(X_train)} sequences...")
    
    # Build LSTM model
    dl_model = Sequential()
    dl_model.add(LSTM(64, input_shape=(seq_length, len(features)), return_sequences=True))
    dl_model.add(Dropout(0.2))
    dl_model.add(LSTM(32))
    dl_model.add(Dropout(0.2))
    dl_model.add(Dense(1))
    
    dl_model.compile(optimizer='adam', loss='mse')
    
    # Early stopping to prevent overfitting
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True)
    
    # Train the model
    history = dl_model.fit(
        X_train, y_train,
        epochs=50,  # Reduced for faster execution
        batch_size=32,
        validation_data=(X_test, y_test),
        callbacks=[early_stopping],
        verbose=1
    )
    
    # Function to inverse transform a single column
    def inverse_transform_single(values, col_idx):
        # Create a dummy array with zeros
        dummy = np.zeros((len(values), scaler.n_features_in_))
        # Set the target column
        dummy[:, col_idx] = values.flatten()
        # Inverse transform
        dummy_inv = scaler.inverse_transform(dummy)
        # Extract the target column
        return dummy_inv[:, col_idx]
    
    # Make predictions on test set
    y_pred = dl_model.predict(X_test)
    
    # Inverse transform for evaluation
    y_test_inv = inverse_transform_single(y_test, 3)
    y_pred_inv = inverse_transform_single(y_pred, 3)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test_inv, y_pred_inv)
    rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
    r2 = r2_score(y_test_inv, y_pred_inv)
    
    print(f"\nDeep Learning Model Evaluation:")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.4f}")
    
    # Plot actual vs predicted
    plt.figure(figsize=(14, 8))
    plt.plot(daily_data['Date'][train_size+seq_length:], y_test_inv, 'b-', label='Actual')
    plt.plot(daily_data['Date'][train_size+seq_length:], y_pred_inv, 'r-', label='Predicted')
    plt.title('Deep Learning Model: Actual vs Predicted Active Cases')
    plt.xlabel('Date')
    plt.ylabel('Active Cases')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Hybrid Model Prediction for Target Date
    print(f"\nGenerating predictions for {target_date}...")
    
    # Define weights for hybrid model
    sir_weight = 0.4  # Weight for SIR model predictions
    dl_weight = 0.6   # Weight for deep learning predictions
    
    # Calculate days between last data point and target date
    latest_date = daily_data['Date'].iloc[-1]
    prediction_days = (target_date - latest_date.to_pydatetime().date()).days
    
    if prediction_days <= 0:
        print(f"WARNING: Target date {target_date} is not in the future. Using next day instead.")
        prediction_days = 1
    
    print(f"Making prediction for {prediction_days} days ahead from {latest_date.strftime('%Y-%m-%d')}")
    
    # Initialize prediction results
    prediction_dates = [latest_date + timedelta(days=i+1) for i in range(prediction_days)]
    predictions = []
    
    # Get last state
    current_S = S0
    current_I = I0
    current_R = R0
    
    latest_confirmed = daily_data['Confirmed'].iloc[-1]
    latest_deaths = daily_data['Deaths'].iloc[-1]
    latest_recovered = daily_data['Recovered'].iloc[-1]
    
    # Get deep learning input
    recent_data = daily_data[features].tail(seq_length).values
    scaled_recent = scaler.transform(recent_data)
    
    # Generate predictions day by day
    for i in range(prediction_days):
        # SIR prediction for next day
        next_day_solution = solve_ivp(
            fun=lambda t, y: sir_model(t, y, beta_opt, gamma_opt, population),
            t_span=[0, 1],
            y0=[current_S, current_I, current_R],
            t_eval=[0, 1],
            method='RK45'
        )
        
        S_sir, I_sir, R_sir = next_day_solution.y
        sir_pred_active = I_sir[1]  # Next day prediction
        
        # Deep learning prediction for next day
        dl_input = scaled_recent.reshape(1, seq_length, len(features))
        dl_pred_scaled = dl_model.predict(dl_input)[0][0]
        dl_pred_active = inverse_transform_single(np.array([dl_pred_scaled]), 3)[0]
        
        # Combine predictions
        hybrid_pred_active = sir_weight * sir_pred_active + dl_weight * dl_pred_active
        
        # Calculate other metrics
        if latest_confirmed > 0:
            death_rate = latest_deaths / latest_confirmed
            recovery_rate = latest_recovered / latest_confirmed
        else:
            death_rate = 0
            recovery_rate = 0
        
        # New cases estimate (difference between current active and predicted)
        new_cases = max(0, hybrid_pred_active - current_I)
        
        # Update total counts
        next_confirmed = latest_confirmed + new_cases
        next_deaths = latest_deaths + (new_cases * death_rate)
        next_recovered = latest_recovered + (new_cases * recovery_rate)
        
        # Store predictions
        prediction = {
            'Date': prediction_dates[i],
            'Confirmed': next_confirmed,
            'Active': hybrid_pred_active,
            'Deaths': next_deaths,
            'Recovered': next_recovered,
            'SIR_Active': sir_pred_active,
            'DL_Active': dl_pred_active
        }
        predictions.append(prediction)
        
        # Update for next iteration
        current_S = S_sir[1]
        current_I = hybrid_pred_active
        current_R = R_sir[1]
        
        latest_confirmed = next_confirmed
        latest_deaths = next_deaths
        latest_recovered = next_recovered
        
        # Update deep learning input for next prediction
        # Create new row with predicted values
        new_row = np.array([[
            next_confirmed, 
            next_deaths, 
            next_recovered, 
            hybrid_pred_active,
            new_cases,  # New_Confirmed
            next_deaths - daily_data['Deaths'].iloc[-1] if i == 0 else next_deaths - predictions[i-1]['Deaths'],  # New_Deaths
            next_recovered - daily_data['Recovered'].iloc[-1] if i == 0 else next_recovered - predictions[i-1]['Recovered']  # New_Recovered
        ]])
        
        # Scale the new row
        scaled_new_row = scaler.transform(new_row)
        
        # Update the sequence by removing the first day and adding the new prediction
        scaled_recent = np.vstack([scaled_recent[1:], scaled_new_row])
    
    # Create predictions dataframe
    pred_df = pd.DataFrame(predictions)
    
    # Display final prediction
    final_pred = predictions[-1]
    print("\n--- Prediction Results for", target_date, "---")
    print(f"Confirmed Cases: {int(final_pred['Confirmed'])}")
    print(f"Active Cases: {int(final_pred['Active'])}")
    print(f"Deaths: {int(final_pred['Deaths'])}")
    print(f"Recovered: {int(final_pred['Recovered'])}")
    print("\nModel Contributions:")
    print(f"SIR Model Active Cases: {int(final_pred['SIR_Active'])}")
    print(f"Deep Learning Active Cases: {int(final_pred['DL_Active'])}")
    
    # Visualize predictions
    plt.figure(figsize=(15, 10))
    
    # Plot historical and predicted data
    plt.subplot(2, 1, 1)
    # Historical
    plt.plot(daily_data['Date'], daily_data['Active'], 'b-', label='Historical Active')
    plt.plot(daily_data['Date'], daily_data['Deaths'], 'r-', label='Historical Deaths')
    plt.plot(daily_data['Date'], daily_data['Recovered'], 'g-', label='Historical Recovered')
    
    # Predictions
    plt.plot(pred_df['Date'], pred_df['Active'], 'b--', label='Predicted Active')
    plt.plot(pred_df['Date'], pred_df['Deaths'], 'r--', label='Predicted Deaths')
    plt.plot(pred_df['Date'], pred_df['Recovered'], 'g--', label='Predicted Recovered')
    
    # Highlight transition point
    plt.axvline(x=latest_date, color='k', linestyle='--', alpha=0.3, label='Last Historical Day')
    
    plt.title(f'COVID-19 Statistics and Predictions for {country}')
    plt.xlabel('Date')
    plt.ylabel('Cases')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    
    # Plot model contributions for final prediction
    plt.subplot(2, 1, 2)
    plt.bar(['SIR Model', 'Deep Learning', 'Hybrid'], 
            [final_pred['SIR_Active'], final_pred['DL_Active'], final_pred['Active']])
    plt.title(f'Model Contributions to Active Cases Prediction for {target_date}')
    plt.ylabel('Predicted Active Cases')
    plt.grid(True, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Return full prediction dataframe
    display(pred_df)


In [8]:
run_button.on_click(run_analysis)